# Персональная рекомендация баннеров: Jupyter/Colab notebook

Этот notebook обучает первую рабочую модель для задачи **показа наиболее интересных баннеров конкретному пользователю**.

Что внутри:
- загрузка CSV из текущей папки,
- объединение пользовательских и баннерных признаков,
- построение исторических CTR-фич **без утечки из будущего**,
- обучение `CatBoostClassifier`,
- оценка на **time-based split**,
- top-K рекомендации для выбранного `user_id`.

> Этот notebook рассчитан на ваши файлы: `banner_interactions.csv`, `users.csv`, `banners.csv`.

## 1) Установка зависимостей

В Colab обычно достаточно выполнить следующую ячейку один раз. Если пакеты уже стоят, шаг можно пропустить.

In [ ]:
# Если запускаете в Colab / чистом окружении, раскомментируйте строку ниже:
# %pip install -q pandas scikit-learn catboost

## 2) Импорты и параметры

Здесь можно поменять путь к данным, количество итераций, `user_id` для рекомендаций и размер top-K.

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import log_loss, roc_auc_score

DATA_DIR = Path('.')          # В Colab можно заменить, например, на Path('/content')
OUTPUT_DIR = Path('./banner_recommender_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ITERATIONS = 300
USER_ID_TO_SCORE = 'u_00001'
TOP_K = 10
GLOBAL_PRIOR_CTR = 0.03

## 3) Вспомогательные функции

Ниже — весь пайплайн: загрузка данных, сбор признаков, time-based split, обучение и функция рекомендаций.

In [ ]:
@dataclass
class DatasetBundle:
    interactions: pd.DataFrame
    users: pd.DataFrame
    banners: pd.DataFrame
    df: pd.DataFrame
    feature_cols: List[str]
    cat_cols: List[str]


def load_data(data_dir: str | Path) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    data_dir = Path(data_dir)
    interactions = pd.read_csv(data_dir / 'banner_interactions.csv', parse_dates=['event_date'])
    users = pd.read_csv(data_dir / 'users.csv')
    banners = pd.read_csv(data_dir / 'banners.csv', parse_dates=['created_at'])
    return interactions, users, banners


def _add_cum_stats(df: pd.DataFrame, group_cols: List[str], prefix: str, prior_weight: int = 20) -> pd.DataFrame:
    grp = df.groupby(group_cols, sort=False)
    prev_clicks = grp['clicks'].cumsum() - df['clicks']
    prev_impr = grp['impressions'].cumsum() - df['impressions']
    df[f'{prefix}_prev_clicks'] = prev_clicks.astype('float32')
    df[f'{prefix}_prev_impr'] = prev_impr.astype('float32')
    df[f'{prefix}_prev_ctr'] = ((prev_clicks + prior_weight * GLOBAL_PRIOR_CTR) / (prev_impr + prior_weight)).astype('float32')
    return df


def build_training_frame(interactions: pd.DataFrame, users: pd.DataFrame, banners: pd.DataFrame) -> DatasetBundle:
    df = interactions.merge(users, on='user_id', how='left').merge(banners, on='banner_id', how='left')
    df = df.sort_values(['event_date', 'user_id', 'banner_id']).reset_index(drop=True)

    df['target'] = (df['clicks'] > 0).astype('int8')
    df['weekday'] = df['event_date'].dt.weekday.astype('int8')
    df['month_day'] = df['event_date'].dt.day.astype('int8')
    df['banner_age_days'] = (df['event_date'] - df['created_at']).dt.days.clip(lower=0).fillna(0).astype('int16')

    for col in ['interest_1', 'interest_2', 'interest_3']:
        df[f'match_{col}'] = (df[col] == df['subcategory']).astype('int8')
    df['match_any_interest'] = df[['match_interest_1', 'match_interest_2', 'match_interest_3']].max(axis=1).astype('int8')
    df['gender_match'] = ((df['target_gender'] == 'U') | (df['gender'] == 'U') | (df['gender'] == df['target_gender'])).astype('int8')
    df['age_in_target'] = ((df['age'] >= df['target_age_min']) & (df['age'] <= df['target_age_max'])).astype('int8')

    stat_defs = [
        (['user_id'], 'user'),
        (['banner_id'], 'banner'),
        (['user_id', 'category'], 'user_cat'),
        (['user_id', 'subcategory'], 'user_subcat'),
        (['category'], 'cat'),
        (['subcategory'], 'subcat'),
        (['brand'], 'brand_hist'),
    ]
    for keys, prefix in stat_defs:
        df = _add_cum_stats(df, keys, prefix)

    feature_cols = [
        'age', 'gender', 'city_tier', 'device_os', 'platform', 'income_band', 'activity_segment',
        'interest_1', 'interest_2', 'interest_3', 'signup_days_ago', 'is_premium',
        'brand', 'category', 'subcategory', 'banner_format', 'campaign_goal', 'target_gender',
        'target_age_min', 'target_age_max', 'cpm_bid', 'quality_score', 'is_active', 'banner_age_days',
        'weekday', 'month_day', 'match_interest_1', 'match_interest_2', 'match_interest_3',
        'match_any_interest', 'gender_match', 'age_in_target',
        'user_prev_impr', 'user_prev_ctr', 'banner_prev_impr', 'banner_prev_ctr',
        'user_cat_prev_impr', 'user_cat_prev_ctr', 'user_subcat_prev_impr', 'user_subcat_prev_ctr',
        'cat_prev_impr', 'cat_prev_ctr', 'subcat_prev_impr', 'subcat_prev_ctr', 'brand_hist_prev_ctr',
    ]

    cat_cols = [
        'gender', 'city_tier', 'device_os', 'platform', 'income_band', 'activity_segment',
        'interest_1', 'interest_2', 'interest_3', 'brand', 'category', 'subcategory',
        'banner_format', 'campaign_goal', 'target_gender',
    ]

    return DatasetBundle(interactions=interactions, users=users, banners=banners, df=df, feature_cols=feature_cols, cat_cols=cat_cols)


def train_valid_test_split(df: pd.DataFrame, valid_days: int = 7, test_days: int = 7):
    max_date = df['event_date'].max()
    test_start = max_date - pd.Timedelta(days=test_days - 1)
    valid_start = test_start - pd.Timedelta(days=valid_days)

    train = df[df['event_date'] < valid_start].copy()
    valid = df[(df['event_date'] >= valid_start) & (df['event_date'] < test_start)].copy()
    test = df[df['event_date'] >= test_start].copy()
    return train, valid, test


def fit_model(bundle: DatasetBundle, iterations: int = 300):
    train, valid, test = train_valid_test_split(bundle.df)

    model = CatBoostClassifier(
        loss_function='Logloss',
        eval_metric='AUC',
        iterations=iterations,
        learning_rate=0.08,
        depth=8,
        l2_leaf_reg=8,
        random_seed=42,
        verbose=50,
    )

    model.fit(
        train[bundle.feature_cols],
        train['target'],
        cat_features=bundle.cat_cols,
        sample_weight=train['impressions'],
        eval_set=(valid[bundle.feature_cols], valid['target']),
        use_best_model=True,
    )

    metrics = {
        'train_rows': int(len(train)),
        'valid_rows': int(len(valid)),
        'test_rows': int(len(test)),
        'date_range': [str(bundle.df['event_date'].min().date()), str(bundle.df['event_date'].max().date())],
    }

    for split_name, part in [('valid', valid), ('test', test)]:
        pred = model.predict_proba(part[bundle.feature_cols])[:, 1]
        metrics[f'{split_name}_auc'] = float(roc_auc_score(part['target'], pred, sample_weight=part['impressions']))
        metrics[f'{split_name}_logloss'] = float(log_loss(part['target'], pred, sample_weight=part['impressions']))
        base_ctr = float(part['clicks'].sum() / part['impressions'].sum())
        metrics[f'{split_name}_base_ctr'] = base_ctr

        ranked = part[['event_date', 'user_id', 'clicks', 'impressions']].copy()
        ranked['pred'] = pred
        for k in (1, 3):
            topk = (
                ranked.sort_values(['event_date', 'user_id', 'pred'], ascending=[True, True, False])
                .groupby(['event_date', 'user_id'], as_index=False)
                .head(k)
            )
            ctr = float(topk['clicks'].sum() / topk['impressions'].sum())
            metrics[f'{split_name}_top{k}_ctr'] = ctr
            metrics[f'{split_name}_top{k}_uplift_vs_base'] = float(ctr / base_ctr) if base_ctr > 0 else None

    return model, metrics


def build_history_snapshots(interactions: pd.DataFrame, banners: pd.DataFrame, scoring_date: pd.Timestamp):
    hist = interactions[interactions['event_date'] < scoring_date].copy()
    hist = hist.merge(banners[['banner_id', 'category', 'subcategory', 'brand']], on='banner_id', how='left')

    user_stats = hist.groupby('user_id', as_index=False).agg(user_clicks=('clicks', 'sum'), user_impr=('impressions', 'sum'))
    user_stats['user_prev_ctr'] = (user_stats['user_clicks'] + 20 * GLOBAL_PRIOR_CTR) / (user_stats['user_impr'] + 20)

    banner_stats = hist.groupby('banner_id', as_index=False).agg(banner_clicks=('clicks', 'sum'), banner_impr=('impressions', 'sum'))
    banner_stats['banner_prev_ctr'] = (banner_stats['banner_clicks'] + 20 * GLOBAL_PRIOR_CTR) / (banner_stats['banner_impr'] + 20)

    uc_stats = hist.groupby(['user_id', 'category'], as_index=False).agg(user_cat_clicks=('clicks', 'sum'), user_cat_impr=('impressions', 'sum'))
    uc_stats['user_cat_prev_ctr'] = (uc_stats['user_cat_clicks'] + 20 * GLOBAL_PRIOR_CTR) / (uc_stats['user_cat_impr'] + 20)

    us_stats = hist.groupby(['user_id', 'subcategory'], as_index=False).agg(user_subcat_clicks=('clicks', 'sum'), user_subcat_impr=('impressions', 'sum'))
    us_stats['user_subcat_prev_ctr'] = (us_stats['user_subcat_clicks'] + 20 * GLOBAL_PRIOR_CTR) / (us_stats['user_subcat_impr'] + 20)

    cat_stats = hist.groupby('category', as_index=False).agg(cat_clicks=('clicks', 'sum'), cat_impr=('impressions', 'sum'))
    cat_stats['cat_prev_ctr'] = (cat_stats['cat_clicks'] + 20 * GLOBAL_PRIOR_CTR) / (cat_stats['cat_impr'] + 20)

    subcat_stats = hist.groupby('subcategory', as_index=False).agg(subcat_clicks=('clicks', 'sum'), subcat_impr=('impressions', 'sum'))
    subcat_stats['subcat_prev_ctr'] = (subcat_stats['subcat_clicks'] + 20 * GLOBAL_PRIOR_CTR) / (subcat_stats['subcat_impr'] + 20)

    brand_stats = hist.groupby('brand', as_index=False).agg(brand_clicks=('clicks', 'sum'), brand_impr=('impressions', 'sum'))
    brand_stats['brand_hist_prev_ctr'] = (brand_stats['brand_clicks'] + 20 * GLOBAL_PRIOR_CTR) / (brand_stats['brand_impr'] + 20)

    return user_stats, banner_stats, uc_stats, us_stats, cat_stats, subcat_stats, brand_stats


def recommend_for_user(user_id: str, scoring_date: pd.Timestamp, model: CatBoostClassifier, bundle: DatasetBundle, top_k: int = 10) -> pd.DataFrame:
    user_row = bundle.users[bundle.users['user_id'] == user_id].copy()
    if user_row.empty:
        raise ValueError(f'Unknown user_id: {user_id}')

    active_banners = bundle.banners[(bundle.banners['is_active'] == 1) & (bundle.banners['created_at'] <= scoring_date)].copy()
    if active_banners.empty:
        raise ValueError('No active banners available for the requested scoring date')

    candidates = active_banners.assign(_tmp_key=1).merge(user_row.assign(_tmp_key=1), on='_tmp_key').drop(columns='_tmp_key')
    candidates['weekday'] = scoring_date.weekday()
    candidates['month_day'] = scoring_date.day
    candidates['banner_age_days'] = (scoring_date - candidates['created_at']).dt.days.clip(lower=0).fillna(0)

    for col in ['interest_1', 'interest_2', 'interest_3']:
        candidates[f'match_{col}'] = (candidates[col] == candidates['subcategory']).astype('int8')
    candidates['match_any_interest'] = candidates[['match_interest_1', 'match_interest_2', 'match_interest_3']].max(axis=1).astype('int8')
    candidates['gender_match'] = ((candidates['target_gender'] == 'U') | (candidates['gender'] == 'U') | (candidates['gender'] == candidates['target_gender'])).astype('int8')
    candidates['age_in_target'] = ((candidates['age'] >= candidates['target_age_min']) & (candidates['age'] <= candidates['target_age_max'])).astype('int8')

    user_stats, banner_stats, uc_stats, us_stats, cat_stats, subcat_stats, brand_stats = build_history_snapshots(
        bundle.interactions, bundle.banners, scoring_date
    )

    candidates = candidates.merge(user_stats[['user_id', 'user_impr', 'user_prev_ctr']], on='user_id', how='left')
    candidates = candidates.rename(columns={'user_impr': 'user_prev_impr'})

    candidates = candidates.merge(banner_stats[['banner_id', 'banner_impr', 'banner_prev_ctr']], on='banner_id', how='left')
    candidates = candidates.rename(columns={'banner_impr': 'banner_prev_impr'})

    candidates = candidates.merge(uc_stats[['user_id', 'category', 'user_cat_impr', 'user_cat_prev_ctr']], on=['user_id', 'category'], how='left')
    candidates = candidates.rename(columns={'user_cat_impr': 'user_cat_prev_impr'})

    candidates = candidates.merge(us_stats[['user_id', 'subcategory', 'user_subcat_impr', 'user_subcat_prev_ctr']], on=['user_id', 'subcategory'], how='left')
    candidates = candidates.rename(columns={'user_subcat_impr': 'user_subcat_prev_impr'})

    candidates = candidates.merge(cat_stats[['category', 'cat_impr', 'cat_prev_ctr']], on='category', how='left')
    candidates = candidates.rename(columns={'cat_impr': 'cat_prev_impr'})

    candidates = candidates.merge(subcat_stats[['subcategory', 'subcat_impr', 'subcat_prev_ctr']], on='subcategory', how='left')
    candidates = candidates.rename(columns={'subcat_impr': 'subcat_prev_impr'})

    candidates = candidates.merge(brand_stats[['brand', 'brand_hist_prev_ctr']], on='brand', how='left')

    default_fill = {
        'user_prev_impr': 0, 'user_prev_ctr': GLOBAL_PRIOR_CTR,
        'banner_prev_impr': 0, 'banner_prev_ctr': GLOBAL_PRIOR_CTR,
        'user_cat_prev_impr': 0, 'user_cat_prev_ctr': GLOBAL_PRIOR_CTR,
        'user_subcat_prev_impr': 0, 'user_subcat_prev_ctr': GLOBAL_PRIOR_CTR,
        'cat_prev_impr': 0, 'cat_prev_ctr': GLOBAL_PRIOR_CTR,
        'subcat_prev_impr': 0, 'subcat_prev_ctr': GLOBAL_PRIOR_CTR,
        'brand_hist_prev_ctr': GLOBAL_PRIOR_CTR,
    }
    candidates = candidates.fillna(default_fill)

    for col in bundle.cat_cols:
        candidates[col] = candidates[col].astype(str)

    candidates['score'] = model.predict_proba(candidates[bundle.feature_cols])[:, 1]
    keep_cols = [
        'user_id', 'banner_id', 'score', 'brand', 'category', 'subcategory', 'banner_format', 'campaign_goal',
        'quality_score', 'cpm_bid', 'match_any_interest', 'user_prev_ctr', 'banner_prev_ctr', 'user_subcat_prev_ctr',
    ]
    return candidates[keep_cols].sort_values('score', ascending=False).head(top_k).reset_index(drop=True)

## 4) Загрузка и быстрый sanity check

Эта ячейка покажет размеры таблиц, период данных и базовый CTR.

In [ ]:
interactions, users, banners = load_data(DATA_DIR)

print('interactions:', interactions.shape)
print('users:', users.shape)
print('banners:', banners.shape)
print('date range:', interactions['event_date'].min().date(), '->', interactions['event_date'].max().date())
print('base CTR:', round(interactions['clicks'].sum() / interactions['impressions'].sum(), 6))

interactions.head()

## 5) Подготовка train table

Здесь строится итоговая таблица обучения и добавляются исторические признаки.

In [ ]:
bundle = build_training_frame(interactions, users, banners)
print('training frame shape:', bundle.df.shape)
print('feature count:', len(bundle.feature_cols))
print('categorical feature count:', len(bundle.cat_cols))

bundle.df[bundle.feature_cols + ['target']].head()

## 6) Обучение модели и метрики

Используем time-based split: train → valid → test по датам, а не random split.

In [ ]:
model, metrics = fit_model(bundle, iterations=ITERATIONS)
print(json.dumps(metrics, ensure_ascii=False, indent=2))

feature_importance = pd.DataFrame({
    'feature': bundle.feature_cols,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False)

feature_importance.head(20)

## 7) Сохранение артефактов

Сохраняем обученную модель, feature importance, метрики и рекомендации.

In [ ]:
model.save_model(str(OUTPUT_DIR / 'catboost_banner_ranker.cbm'))
feature_importance.to_csv(OUTPUT_DIR / 'feature_importance.csv', index=False)
with open(OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('saved to:', OUTPUT_DIR.resolve())

## 8) Рекомендации для конкретного пользователя

Ниже можно поменять `USER_ID_TO_SCORE` и получить top-K баннеров на следующий день после последней даты в train-таблице.

In [ ]:
scoring_date = bundle.df['event_date'].max() + pd.Timedelta(days=1)
recommendations = recommend_for_user(USER_ID_TO_SCORE, scoring_date, model, bundle, top_k=TOP_K)
recommendations.to_csv(OUTPUT_DIR / f'recommendations_{USER_ID_TO_SCORE}.csv', index=False)
recommendations

## 9) Что улучшать дальше

После первого baseline рекомендую развивать решение в таком порядке:

1. Добавить больше history-features по окнам: 3/7/14/30 дней.  
2. Ввести frequency capping и фильтр уже часто показанных баннеров.  
3. Перейти от простой CTR-модели к **ranking / contextual bandit** для online-serving.  
4. Валидировать не только AUC, но и **top-K CTR uplift** и online A/B test.  
5. Добавить revenue-aware objective, если важно не только клик, но и доход / conversion.

Если захотите, следующий шаг — сделать вторую версию notebook с:
- `top_k` метриками подробнее,
- negative sampling / candidate generation,
- простым API-скриптом для online scoring.